# AIC 2026 — Whisper Medium ASR Worker

Notebook này chạy **M2-ASR pilot** bằng `faster-whisper medium`, tiếng Việt.

Nó làm đúng 1 việc: nhận một shard manifest, chạy Whisper cho các video trong shard, lưu transcript có timestamp, validate, rồi zip artifact.

Không tạo ground truth, không chạy CLIP, không rebuild M1.

Quy tắc:

- Chạy smoke 5 video trước.
- Smoke và full run phải dùng output directory khác nhau.
- Nếu runtime bị ngắt, chạy lại đúng shard + đúng output dir để resume.
- `DONE.json` chỉ được ghi khi shard hoàn tất hợp lệ.

## Kaggle — bản đã chỉnh rõ setup

Kaggle **không đọc trực tiếp được Google Drive MyDrive**. Vì vậy data video và shard phải được đưa vào Kaggle dưới dạng Dataset/Input, hoặc upload thủ công.

Repo được clone từ GitHub:

```text
https://github.com/doraIaIa/aic2026.git
```

Kaggle mặc định chạy `shard_001_of_002.json`.


In [ ]:
# ===== 1) User config for Kaggle =====
REPO_URL = "https://github.com/doraIaIa/aic2026.git"
REPO_DIR = "/kaggle/working/aic2026"

# Bạn phải tạo/thêm Kaggle Dataset chứa video, rồi sửa VIDEO_ROOT cho đúng.
# Ví dụ dataset có cấu trúc: /kaggle/input/aic-2026-data/data_extracted/video
VIDEO_ROOT = "/kaggle/input/aic-2026-data/data_extracted/video"

# Dataset chứa shard_001_of_002.json.
SHARD_PATH = "/kaggle/input/aic-asr-shards/shard_001_of_002.json"
OUTPUT_ROOT = "/kaggle/working/asr-worker/artifacts/whisper-medium-vi-pilot-v1"
CONFIG_PATH = "/kaggle/working/asr-worker/aic-runtime.toml"

WORKER = "kaggle"
NUM_SHARDS = 2
SHARD_INDEX = 1
SHARD_NAME = "shard_001_of_002"

MODEL = "medium"
MODEL_REVISION = "medium"
LANGUAGE = "vi"
BEAM_SIZE = 5
VAD_FILTER = True
DEVICE = "auto"

SMOKE_LIMIT = 5
RUN_FULL_SHARD = False

print({
    "repo_url": REPO_URL,
    "repo_dir": REPO_DIR,
    "video_root": VIDEO_ROOT,
    "shard_path": SHARD_PATH,
    "output_root": OUTPUT_ROOT,
    "run_full_shard": RUN_FULL_SHARD,
    "smoke_limit": SMOKE_LIMIT,
})


In [ ]:
# ===== 2) Clone/update repo from GitHub =====
from pathlib import Path
import subprocess

Path('/kaggle/working/asr-worker').mkdir(parents=True, exist_ok=True)
repo = Path(REPO_DIR)
if repo.exists():
    print('Repo exists, fetching latest if possible:', REPO_DIR)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--prune'], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'status', '--short'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-5'], check=False)


In [ ]:
# ===== 3) Install dependencies =====
import subprocess, sys

subprocess.run(['apt-get', 'update', '-y'], check=False)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'faster-whisper'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO_DIR}[asr]'], check=True)
print('Install done.')


In [ ]:
# ===== 4) Check GPU and CLI =====
import shutil, subprocess, sys

print('ffmpeg:', shutil.which('ffmpeg'))
subprocess.run(['nvidia-smi'], check=False)
import faster_whisper
print('faster_whisper: OK')
subprocess.run([sys.executable, '-m', 'aic2026.cli', '--help'], check=True)


In [ ]:
# ===== 5) Validate Kaggle input paths and create config =====
from pathlib import Path

for name, value in {'VIDEO_ROOT': VIDEO_ROOT, 'SHARD_PATH': SHARD_PATH}.items():
    p = Path(value)
    print(name, value, 'exists =', p.exists())

if not Path(VIDEO_ROOT).exists():
    raise FileNotFoundError(
        'Không thấy VIDEO_ROOT. Kaggle không thấy Google Drive; hãy tạo Dataset chứa video và sửa VIDEO_ROOT.'
    )
if not Path(SHARD_PATH).exists():
    raise FileNotFoundError(
        'Không thấy SHARD_PATH. Hãy tạo Dataset chứa shard_001_of_002.json hoặc sửa SHARD_PATH.'
    )

Path(CONFIG_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(CONFIG_PATH).write_text('# Auto-generated Kaggle config
' + f'video_root = "{VIDEO_ROOT}"
', encoding='utf-8')
print('CONFIG_PATH:', CONFIG_PATH)
print(Path(CONFIG_PATH).read_text())


In [ ]:
# ===== 6) Run ASR: smoke first, full shard later =====
from pathlib import Path
import subprocess, sys

if RUN_FULL_SHARD:
    out_dir = Path(OUTPUT_ROOT) / SHARD_NAME
    limit_args = []
else:
    out_dir = Path(OUTPUT_ROOT) / f"smoke_{SHARD_NAME}_limit{SMOKE_LIMIT}"
    limit_args = ['--limit-videos', str(SMOKE_LIMIT)]

out_dir.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, '-m', 'aic2026.cli', 'run-asr-shard',
    '--config', CONFIG_PATH,
    '--shard', SHARD_PATH,
    '--out-dir', str(out_dir),
    '--model', MODEL,
    '--model-revision', MODEL_REVISION,
    '--language', LANGUAGE,
    '--beam-size', str(BEAM_SIZE),
    '--device', DEVICE,
]
if VAD_FILTER:
    cmd.append('--vad-filter')
cmd.extend(limit_args)
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Output dir:', out_dir)


In [ ]:
# ===== 7) Validate, preview, zip output =====
from pathlib import Path
import subprocess, sys, zipfile

if RUN_FULL_SHARD:
    out_dir = Path(OUTPUT_ROOT) / SHARD_NAME
else:
    out_dir = Path(OUTPUT_ROOT) / f"smoke_{SHARD_NAME}_limit{SMOKE_LIMIT}"

subprocess.run([sys.executable, '-m', 'aic2026.cli', 'validate-asr-shard', '--artifact-dir', str(out_dir)], check=True)

for name in ['DONE.json', 'progress.json']:
    p = out_dir / name
    print(f'===== {name} =====')
    print(p.read_text(encoding='utf-8')[:4000] if p.exists() else 'MISSING')

seg = out_dir / 'asr_segments.jsonl'
print('===== first 10 asr_segments.jsonl =====')
if seg.exists():
    with seg.open('r', encoding='utf-8') as f:
        for i, line in zip(range(10), f):
            print(line.rstrip()[:1000])

err = out_dir / 'errors.jsonl'
print('===== first 20 errors.jsonl =====')
if err.exists():
    with err.open('r', encoding='utf-8') as f:
        for i, line in zip(range(20), f):
            print(line.rstrip()[:1000])

zip_path = out_dir.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in out_dir.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(out_dir.parent))
print('ZIP artifact:', zip_path)
